# SYNORA: one genuine ACE-Step GPU experiment
**Not executed.** Use a permitted local/university GPU or manually selected interactive free notebook only. No hosted API, rental, quota bypass, customer audio or automated scheduling. Stop if charges would apply. Open this notebook from a checked-out Synora repository; set SYNORA to its path. The pinned upstream dependencies and ~10 GB snapshot need substantial disk space. CUDA 12.8-compatible driver, preferably 24 GB VRAM. Review all licenses and hosting terms before enabling downloads. Notebook does not allocate compute itself.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, json
SYNORA = Path.cwd().resolve()  # change to Synora repo root if needed
WORK = SYNORA.parent / "synora-ace-local-experiment"
ACE = WORK / "ace-pinned"
OUTPUT = WORK / "one-song"
CODE = "ca1e85fe9430179831e6bc6be790c332190a3866"
WEIGHTS = "eb84216a8ece2d56388e401efc28061a0c30a209"
ALLOW_AUTHORIZED_FREE_DOWNLOADS = False  # change only after reviewing terms, permissions and no-charge limits
assert (SYNORA / "benchmarks/music/ace_interactive.py").is_file(), "Set SYNORA correctly"
assert os.getenv("SYNORA_ENV") != "production"
for tool in ("git", "uv", "ffmpeg", "ffprobe", "nvidia-smi"):
    assert shutil.which(tool), f"Missing {tool}"
subprocess.run(["nvidia-smi"], check=True)
assert shutil.disk_usage(SYNORA.parent).free >= 35 * 1024**3, "Insufficient disk"
print("Hardware precheck only; no downloads or generation executed.")

In [ ]:
assert ALLOW_AUTHORIZED_FREE_DOWNLOADS is True, "Manual authorization required"
assert not WORK.exists(), "Choose a fresh workspace; no automatic reuse"
WORK.mkdir(mode=0o700)
subprocess.run(["git", "clone", "https://github.com/ace-step/ACE-Step-1.5.git", str(ACE)], check=True)
subprocess.run(["git", "-C", str(ACE), "checkout", "--detach", CODE], check=True)
subprocess.run(["uv", "sync", "--locked", "--python", "3.12"], cwd=ACE, check=True)
PY = ACE / ".venv/bin/python"
env = dict(os.environ, PYTHONPATH=str(SYNORA), SYNORA_ENV="development")
subprocess.run([str(PY), "-m", "benchmarks.music.ace_evidence"], cwd=SYNORA, env=env, check=True)
SNAPSHOT = subprocess.check_output([str(PY), "-c", "from huggingface_hub import snapshot_download; print(snapshot_download(repo_id='ACE-Step/Ace-Step1.5', revision='" + WEIGHTS + "'))"], cwd=ACE, text=True).strip()
assert Path(SNAPSHOT).name == WEIGHTS
print("Pinned snapshot prepared; inference has not executed.")

## Before inference
Disconnect network or use host-supported no-network execution after downloads. Offline environment variables alone do not enforce isolation. Confirm the free allocation has enough remaining time and no overage. Do not continue if the notebook environment cannot honor those requirements.

In [ ]:
CONFIRM_OFFLINE_NO_CHARGE_INFERENCE = False  # manual confirmation after downloads
assert ALLOW_AUTHORIZED_FREE_DOWNLOADS and CONFIRM_OFFLINE_NO_CHARGE_INFERENCE
assert not OUTPUT.exists(), "Do not overwrite evidence"
subprocess.run([str(PY), "-m", "benchmarks.music.ace_interactive", "--ace-source", str(ACE), "--snapshot", SNAPSHOT, "--output", str(OUTPUT), "--authorized-local-gpu"], cwd=SYNORA, env=dict(env, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1"), check=True)

In [ ]:
subprocess.run([str(PY), "-m", "benchmarks.music.ace_evidence", "--verify", str(OUTPUT)], cwd=SYNORA, env=env, check=True)
report = json.loads((OUTPUT / "experiment.json").read_text())
from IPython.display import Audio, display
display(Audio(filename=str(OUTPUT / report["artifact"])))
print("Artifact integrity checked; origin and musical quality require independent review.")
# Keep audio and experiment.json private. No automatic upload or publishing.